# **Dynamic Time Wrapping**





## **Import Library dan Set Path**

---


Kode tersebut digunakan untuk menetapkan konfigurasi awal yang diperlukan dalam proses pemuatan, ekstraksi fitur, dan pembandingan berkas audio dengan menggunakan MFCC dan Dynamic Time Warping. Beragam library seperti os, librosa, numpy, pandas, glob, dan tqdm diimpor untuk mendukung pengelolaan sistem file, pemrosesan sinyal audio, perhitungan numerik, pengolahan data berbentuk tabel, pencarian berkas dengan pola tertentu, serta penampilan progress bar. Variabel SR (sampling rate) diatur pada 22050 Hz sebagai standar kualitas pembacaan audio, sementara N_MFCC ditentukan sebanyak 13 untuk mengambil 13 nilai koefisien MFCC pada setiap berkas audio. Struktur OLD_DIRS menyimpan lokasi direktori audio lama untuk kategori “buka” dan “tutup”, sedangkan NEW_FILES menampung lokasi berkas audio baru yang akan dibandingkan. Pada bagian akhir, OUT_CSV menjadi nama file output yang akan digunakan untuk menyimpan hasil perhitungan jarak DTW dalam bentuk CSV.


In [71]:
import os
import librosa
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

SR = 22050
N_MFCC = 13

BASE_DIR = '/mnt/data/irna_unzipped/user1'

OLD_DIRS = {
    'buka': os.path.join(BASE_DIR, 'buka'),
    'tutup': os.path.join(BASE_DIR, 'tutup'),
}

NEW_FILES = {
    'buka_baru': os.path.join(BASE_DIR, 'bukabaru.wav'),
    'tutup_baru': os.path.join(BASE_DIR, 'tutupbaru.wav'),
}

OUT_CSV = 'dtw_results.csv'


## **Fungsi extrac dan DTW**

---



Kode tersebut memuat dua fungsi utama yang digunakan untuk mengolah audio dengan fitur MFCC dan menghitung kesamaan menggunakan Dynamic Time Warping (DTW). Fungsi **extract_mfcc()** berperan membaca file audio dari lokasi tertentu menggunakan *librosa.load* dengan sampling rate yang telah ditetapkan, kemudian mengonversinya menjadi mono. Setelah itu, fungsi mengekstraksi ciri suara berupa matriks MFCC melalui *librosa.feature.mfcc*.

Fungsi kedua, **dtw_distance()**, menerima dua matriks MFCC (m1 dan m2) dan menghitung seberapa mirip keduanya dengan algoritma DTW dari *librosa.sequence.dtw*. Proses ini menghasilkan matriks biaya kumulatif D serta jalur pencocokan wp. Nilai jarak asli diperoleh dari elemen terakhir matriks D, lalu dinormalisasi dengan membaginya terhadap panjang jalur DTW. Fungsi ini akhirnya mengembalikan jarak mentah, jarak yang telah dinormalisasi, serta panjang jalur sebagai indikator tingkat kemiripan antara dua sinyal audio.


In [72]:
def extract_mfcc(path, sr=SR, n_mfcc=N_MFCC):
    # Load audio dari folder hasil ekstraksi ZIP kamu
    y, _ = librosa.load(path, sr=sr, mono=True)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return mfcc


def dtw_distance(m1, m2, metric='euclidean'):
    # Hitung DTW antara 2 MFCC
    D, wp = librosa.sequence.dtw(X=m1, Y=m2, metric=metric)

    raw = D[-1, -1]         # total cost
    path_len = len(wp)      # panjang lintasan
    norm = raw / path_len if path_len > 0 else np.inf  # normalisasi

    return float(raw), float(norm), int(path_len)


## **Collect data lama dan ekstrak mfcc**

---
Potongan kode pertama digunakan untuk mengumpulkan seluruh file audio lama (.wav) dari folder yang tercantum dalam OLD_DIRS. Dengan memanfaatkan glob, kode menelusuri setiap direktori untuk mencari berkas berekstensi .wav, lalu menyimpannya dalam dictionary old_files sesuai kategorinya, yaitu buka dan tutup. Jika suatu folder tidak memiliki file audio, program akan menampilkan pesan peringatan. Setelah daftar file terkumpul, proses ekstraksi MFCC dilakukan untuk semua audio lama tersebut. Hasil ekstraksi disimpan dalam dictionary old_mfcc, dan setiap file diproses secara berurutan berdasarkan labelnya. tqdm digunakan untuk menampilkan progress bar sehingga prosesnya lebih jelas saat jumlah file yang diproses banyak.

Pada bagian berikutnya, kode melakukan ekstraksi MFCC untuk file audio baru (data uji) yang tercantum dalam NEW_FILES. Sebelum diproses, program memeriksa keberadaan file menggunakan os.path.exists. Jika file tidak ditemukan, program akan menghentikan eksekusi dan memunculkan FileNotFoundError. Apabila file valid, MFCC diekstraksi dengan fungsi extract_mfcc dan hasilnya disimpan dalam dictionary new_mfcc. MFCC dari file baru ini nantinya akan dibandingkan dengan MFCC file lama melalui perhitungan DTW untuk menentukan tingkat kemiripan antara sinyal audio tersebut.


In [9]:
import os

def walk_all(root="/mnt/data"):
    for path, dirs, files in os.walk(root):
        print("PATH :", path)
        print(" DIRS:", dirs)
        print(" FILES:", files)
        print("-" * 60)

walk_all("/mnt/data")

In [10]:
import os

print(os.listdir("/mnt"))
print(os.listdir("/"))


[]
['opt', 'libx32', 'tmp', 'proc', 'dev', 'bin', 'mnt', 'root', 'srv', 'home', 'var', 'etc', 'sbin', 'usr', 'boot', 'sys', 'lib32', 'lib64', 'media', 'lib', 'run', 'kaggle', '.dockerenv', 'datalab', 'tools', 'content', 'python-apt', 'python-apt.tar.xz', 'NGC-DL-CONTAINER-LICENSE', 'cuda-keyring_1.1-1_all.deb']


In [11]:
from google.colab import files
uploaded = files.upload()


Saving user 1 - silvi.zip to user 1 - silvi.zip


In [13]:
import os
print(os.listdir())

['.config', 'user 1 - silvi.zip', 'sample_data']


In [14]:
import zipfile
import os

zip_path = "user 1 - silvi.zip"   # gunakan nama file yang terdeteksi

extract_path = "rekaman_silvi"     # folder tujuan ekstraksi

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Isi folder rekaman_silvi:")
print(os.listdir(extract_path))


Isi folder rekaman_silvi:
['user 1 - silvi']


In [15]:
for path, dirs, files in os.walk("rekaman_silvi"):
    print("PATH :", path)
    print("DIRS :", dirs)
    print("FILES:", files)
    print("-" * 50)


PATH : rekaman_silvi
DIRS : ['user 1 - silvi']
FILES: []
--------------------------------------------------
PATH : rekaman_silvi/user 1 - silvi
DIRS : ['buka', 'tutup']
FILES: []
--------------------------------------------------
PATH : rekaman_silvi/user 1 - silvi/buka
DIRS : []
FILES: ['Recording (57).wav', 'Recording (23).wav', 'Recording (47).wav', 'Recording (82).wav', 'Recording (90).wav', 'Recording (50).wav', 'Recording (98).wav', 'Recording (58).wav', 'Recording (39).wav', 'Recording (43).wav', 'Recording (59).wav', 'Recording (6).wav', 'Recording (25).wav', 'Recording (73).wav', 'Recording (53).wav', 'Recording (24).wav', 'Recording (60).wav', 'Recording (17).wav', 'Recording (14).wav', 'Recording (36).wav', 'Recording (18).wav', 'Recording (52).wav', 'Recording (55).wav', 'Recording (38).wav', 'Recording (13).wav', 'Recording (70).wav', 'Recording (22).wav', 'Recording (15).wav', 'Recording (94).wav', 'Recording (72).wav', 'Recording (9).wav', 'Recording (77).wav', 'Recordin

In [16]:
import os
import librosa
import numpy as np
from glob import glob
from tqdm import tqdm


# ============================
# 1. SET PATH DATA
# ============================

BASE = "rekaman_silvi/user 1 - silvi"

OLD_DIRS = {
    "buka":  os.path.join(BASE, "buka"),
    "tutup": os.path.join(BASE, "tutup"),
}

# Jika ingin menambahkan file baru, taruh file baru di root project,
# lalu isi seperti ini (boleh dikosongkan dulu)
NEW_FILES = {
    # "buka_baru": "buka_baru.wav",
    # "tutup_baru": "tutup_baru.wav"
}


# ============================
# 2. FUNGSI EXTRACT MFCC
# ============================

SR = 16000
N_MFCC = 20

def extract_mfcc(path, sr=SR, n_mfcc=N_MFCC):
    y, _ = librosa.load(path, sr=sr, mono=True)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return mfcc


# ============================
# 3. COLLECT FILE DATA LAMA
# ============================

print("Mengumpulkan daftar file data lama...\n")

old_files = {}
for label, folder in OLD_DIRS.items():

    if not os.path.exists(folder):
        raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")

    pattern = os.path.join(folder, "*.wav")
    files = sorted(glob(pattern))

    print(f"{label} → {len(files)} file ditemukan")
    old_files[label] = files


# ============================
# 4. EKSTRAK MFCC DATA LAMA
# ============================

print("\nEkstrak MFCC untuk data lama...\n")

old_mfcc = {}
for label, files in old_files.items():
    old_mfcc[label] = {}
    for f in tqdm(files, desc=f"MFCC {label}"):
        old_mfcc[label][f] = extract_mfcc(f)


# ============================
# 5. EKSTRAK MFCC UNTUK FILE BARU (Opsional)
# ============================

new_mfcc = {}

if len(NEW_FILES) > 0:
    print("\nEkstrak MFCC untuk file baru...\n")

    for name, path in NEW_FILES.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"File baru tidak ditemukan: {path}")
        new_mfcc[name] = extract_mfcc(path)


print("\nSELESAI! MFCC berhasil diekstrak.")


Mengumpulkan daftar file data lama...

buka → 100 file ditemukan
tutup → 100 file ditemukan

Ekstrak MFCC untuk data lama...



MFCC tutup: 100%|██████████| 100/100 [00:01<00:00, 50.75it/s]


SELESAI! MFCC berhasil diekstrak.


Ekstrak MFCC untuk data lama (buka & tutup)

In [32]:
from glob import glob
from tqdm import tqdm

buka_files = sorted(glob(os.path.join(BUKA_DIR, "*.wav")))
tutup_files = sorted(glob(os.path.join(TUTUP_DIR, "*.wav")))
print("BUKA:", len(buka_files), "TUTUP:", len(tutup_files))

old_mfcc = {"buka":{}, "tutup":{}}
for p in tqdm(buka_files, desc="MFCC buka"):
    m = extract_mfcc_safe(p)
    if m is not None:
        old_mfcc["buka"][os.path.basename(p)] = m

for p in tqdm(tutup_files, desc="MFCC tutup"):
    m = extract_mfcc_safe(p)
    if m is not None:
        old_mfcc["tutup"][os.path.basename(p)] = m

print("Old MFCC counts:", {k: len(v) for k,v in old_mfcc.items()})


BUKA: 100 TUTUP: 100


MFCC tutup: 100%|██████████| 100/100 [00:01<00:00, 51.30it/s]

Old MFCC counts: {'buka': 100, 'tutup': 100}


In [57]:
import os
# Asumsi NEW_DIR sudah terdefinisi dari langkah sebelumnya, misalnya "Rekaman_Baru"

print("--- Status Data Baru ---")
# Cek isi folder rekaman baru
if 'NEW_DIR' in locals() or 'NEW_DIR' in globals():
    print(f"Path Folder Baru (NEW_DIR): {NEW_DIR}")
    if os.path.exists(NEW_DIR):
        files_in_new_dir = [f for f in os.listdir(NEW_DIR) if f.lower().endswith(('.wav', '.mp3', '.flac'))]
        print(f"Jumlah file audio di {NEW_DIR}: {len(files_in_new_dir)}")
        if len(files_in_new_dir) > 0:
            print("Contoh file:", files_in_new_dir[:5])
        else:
            print("⚠️ Folder NEW_DIR KOSONG! Pastikan Anda telah mengupload file audio.")
    else:
        print(f"❌ Folder NEW_DIR tidak ditemukan di path saat ini: {NEW_DIR}")
else:
    print("❌ Variabel NEW_DIR belum terdefinisi. Jalankan ulang seluruh sel setup.")

# Cek hasil ekstraksi MFCC
if 'new_mfcc' in locals() or 'new_mfcc' in globals():
    print(f"\nJumlah MFCC file baru (new_mfcc): {len(new_mfcc)}")
    if len(new_mfcc) == 0:
        print("🛑 WARNING: new_mfcc KOSONG. Ini adalah penyebab utama error Anda.")
    else:
        print("✅ new_mfcc memiliki data. Lanjut ke Perhitungan DTW.")
else:
    print("❌ Variabel new_mfcc belum terdefinisi.")

--- Status Data Baru ---
Path Folder Baru (NEW_DIR): Rekaman_Baru
Jumlah file audio di Rekaman_Baru: 0
⚠️ Folder NEW_DIR KOSONG! Pastikan Anda telah mengupload file audio.

Jumlah MFCC file baru (new_mfcc): 0
🛑 WARNING: new_mfcc KOSONG. Ini adalah penyebab utama error Anda.


## **Perhitungan DTW**


---

Potongan kode ini menjalankan proses utama berupa perhitungan jarak antara data audio baru dan data audio lama menggunakan metode Dynamic Time Warping (DTW). Pertama, disiapkan list kosong bernama *rows* sebagai penampung hasil perhitungan. Melalui tiga lapisan perulangan, setiap fitur audio baru dibandingkan dengan seluruh fitur audio lama sesuai labelnya (buka atau tutup). Proses ini ditampilkan menggunakan *tqdm* agar perkembangan perhitungan terlihat.

Di setiap iterasi, MFCC dari file lama dan baru diproses oleh fungsi `dtw_distance()`, yang menghasilkan tiga output: *raw_cost* (biaya kumulatif), *normalized_cost* (biaya yang telah dinormalisasi), serta *path_len* (panjang lintasan DTW). Seluruh hasil tersebut dirangkum dalam sebuah dictionary dan dimasukkan ke dalam list *rows*.

Setelah semua kombinasi selesai dihitung, *rows* diubah menjadi DataFrame dengan bantuan pandas agar data lebih mudah dianalisis. DataFrame ini kemudian diekspor ke file CSV sesuai nama yang ditentukan dalam variabel `OUT_CSV` (misalnya *dtw_results.csv*). CSV tersebut berisi informasi lengkap mengenai jarak DTW antara setiap file audio baru dan seluruh file audio lama. Pada bagian akhir, kode menampilkan pesan bahwa proses perhitungan telah selesai dan hasilnya berhasil disimpan.


In [22]:
OUT_CSV = "dtw_results.csv"

rows = []
print("Menghitung DTW...")

for new_name, new_feat in new_mfcc.items():
    for label, files in old_files.items():           # label = 'tutup'
        for f in tqdm(files, desc=f"DTW {new_name} vs {label}", leave=False):

            m_old = old_mfcc[label][f]               # ambil MFCC tutup

            raw, norm, plen = dtw_distance(m_old, new_feat)

            rows.append({
                'new_file': new_name,
                'old_label': label,
                'old_path': f,
                'raw_cost': raw,
                'normalized_cost': norm,
                'path_len': plen
            })

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print(f"Hasil tersimpan ke {OUT_CSV}")


Menghitung DTW...
Hasil tersimpan ke dtw_results.csv


In [52]:
import pandas as pd
from tqdm import tqdm
# Asumsi: dtw_distance, new_mfcc, old_files, old_mfcc sudah tersedia

OUT_CSV = "dtw_results.csv"

rows = []
print("Menghitung DTW...")

# Pengecekan: Pastikan ada file baru untuk dibandingkan
if not new_mfcc:
    print("[ERROR] Variabel new_mfcc kosong. Pastikan Anda telah berhasil mengekstrak MFCC untuk file baru.")
else:
    for new_name, new_feat in new_mfcc.items():
        # Iterasi untuk setiap kategori (buka, tutup)
        for label, files in old_files.items():
            # Iterasi untuk setiap file lama dalam kategori tersebut
            for f in tqdm(files, desc=f"DTW {new_name} vs {label}", leave=False):

                try:
                    # Ambil fitur MFCC lama
                    m_old = old_mfcc[label][f]
                except KeyError:
                    # Jika MFCC file lama tidak ditemukan, lewati
                    continue

                if m_old is None:
                    continue

                # Panggil fungsi DTW (raw, normalized, path_len)
                raw, norm, plen = dtw_distance(m_old, new_feat)

                # Kumpulkan hasil
                rows.append({
                    'new_file': new_name,
                    'old_label': label,
                    'old_path': f,
                    'raw_cost': raw,
                    'normalized_cost': norm,
                    'path_len': plen
                })

    # Buat DataFrame dan simpan
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(OUT_CSV, index=False)
        print(f"Hasil perhitungan DTW berhasil disimpan ke {OUT_CSV}")
    else:
        print("[ERROR] List 'rows' kosong. Perhitungan DTW gagal. Cek data input.")

Menghitung DTW...
[ERROR] Variabel new_mfcc kosong. Pastikan Anda telah berhasil mengekstrak MFCC untuk file baru.


## **Kemiripan Data Lama dan Baru**

---

Potongan kode ini berfungsi menjalankan **analisis kemiripan audio** setelah proses Dynamic Time Warping (DTW) selesai, dengan tujuan mengidentifikasi rekaman lama mana yang paling menyerupai setiap rekaman baru. Proses ini dibagi menjadi dua langkah utama. **Langkah 1** berfokus pada **pemuatan data**: ia mencoba memuat hasil perhitungan DTW yang tersimpan dalam file CSV bernama `dtw_results.csv` ke dalam *DataFrame* **`df`** menggunakan pustaka Pandas. Proses ini dilengkapi dengan penanganan *error* untuk memastikan `df` tidak hilang dari memori atau file CSV rusak, memberikan pesan konfirmasi atau peringatan yang sesuai. **Langkah 2** adalah **Analisis dan Tampilan Hasil**: jika *DataFrame* `df` berhasil dimuat dan berisi data yang valid (tidak kosong dan memiliki kolom kunci seperti `'new_file'`), kode ini kemudian melakukan iterasi untuk setiap file audio baru. Untuk setiap file baru, ia akan memfilter dan mengurutkan seluruh hasil DTW berdasarkan kolom **`normalized_cost`** secara *ascending* (biaya terendah), karena biaya DTW yang lebih rendah menandakan **kemiripan yang lebih tinggi**. Akhirnya, kode ini menampilkan **5 rekaman lama teratas** yang paling mirip, memberikan detail label, jalur file, dan skor kemiripan (normalized cost) mereka, menyajikan hasil akhir yang terstruktur.



In [58]:
import pandas as pd
import os

OUT_CSV = "dtw_results.csv"

# =======================================================
# LANGKAH 1: MEMUAT DATA DTW
# =======================================================

# Inisialisasi df (jika belum ada)
df = None

# Cek apakah file hasil DTW ada, lalu muat ke DataFrame
if os.path.exists(OUT_CSV):
    try:
        df = pd.read_csv(OUT_CSV)
        print(f"✅ Data DTW berhasil dimuat dari {OUT_CSV}.")
    except Exception as e:
        print(f"❌ Gagal memuat DataFrame dari CSV: {e}")
        # Jika gagal, df tetap None
else:
    print(f"❌ File hasil perhitungan DTW ('{OUT_CSV}') tidak ditemukan.")
    print("Silakan jalankan ulang semua sel mulai dari ekstraksi MFCC hingga perhitungan DTW.")


# =======================================================
# LANGKAH 2: ANALISIS DAN TAMPILKAN HASIL
# =======================================================

if df is not None and not df.empty and 'new_file' in df.columns:

    print("\n## Hasil Analisis Kemiripan DTW (Top 5 Paling Mirip) 📊")
    print("--------------------------------------------------")

    # Ambil daftar unik dari semua file audio baru
    new_files_list = df['new_file'].unique()

    # Iterasi untuk setiap file baru
    for new_file in new_files_list:

        # 1. Filter subset DataFrame untuk file baru ini
        sub = df[df['new_file'] == new_file]

        # 2. Urutkan berdasarkan normalized_cost (Biaya terkecil = Paling Mirip)
        sub_sorted = sub.sort_values(by='normalized_cost', ascending=True)

        # 3. Ambil 5 baris teratas
        top_5_match = sub_sorted.head(5)

        # 4. Tampilkan hasilnya
        print(f"\n📢 Rekaman Baru: {new_file}")
        print("5 Rekaman Lama Paling Mirip (Normalized Cost Terkecil):")

        # Pilih dan tampilkan kolom yang relevan tanpa index
        print(
            top_5_match[[
                'old_label',
                'old_path',
                'normalized_cost'
            ]].to_string(index=False)
        )

    print("\n--------------------------------------------------")
    print("Analisis selesai.")

elif df is None or df.empty:
    print("\n[PERINGATAN] Analisis dibatalkan karena data DTW (DataFrame 'df') tidak valid atau kosong.")
    print("Pastikan file audio baru sudah ada dan perhitungan DTW telah berhasil dijalankan.")
else:
     print("\n[PERINGATAN] Kolom yang dibutuhkan tidak ada dalam DataFrame.")
     print("Kolom yang ditemukan:", df.columns.tolist())

❌ Gagal memuat DataFrame dari CSV: No columns to parse from file

[PERINGATAN] Analisis dibatalkan karena data DTW (DataFrame 'df') tidak valid atau kosong.
Pastikan file audio baru sudah ada dan perhitungan DTW telah berhasil dijalankan.


# **Visualisasi**

In [74]:
import matplotlib.pyplot as plt

print("\nMenyiapkan visualisasi...")

# Pastikan df punya kolom: old_file, old_label, normalized_cost
summary = df.groupby(['old_label'])['normalized_cost'].mean().reset_index()

# =============================
# 1. BARPLOT rata-rata cost per label lama
# =============================
plt.figure(figsize=(6,4))
plt.bar(summary['old_label'], summary['normalized_cost'])
plt.title("Rata-rata Normalized DTW Cost per Label Lama")
plt.ylabel("Normalized DTW Cost (Lower = Better)")
plt.xlabel("Label Lama")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


# =============================
# 2. HISTOGRAM distribusi cost setiap label lama
# =============================
plt.figure(figsize=(10,4))

for label in df['old_label'].unique():
    subset = df[df['old_label'] == label]['normalized_cost']
    plt.hist(subset, bins=15, alpha=0.5, label=f"{label}")

plt.title("Distribusi Normalized DTW Cost per Label Lama")
plt.xlabel("Normalized DTW Cost")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


# =============================
# 3. SCATTER PLOT cost semua file
# =============================
df_sorted = df.sort_values('normalized_cost')

plt.figure(figsize=(8,4))
plt.scatter(range(len(df_sorted)), df_sorted['normalized_cost'], alpha=0.6)
plt.title("Scatter Plot Normalized DTW Cost")
plt.xlabel("File index (Old files)")
plt.ylabel("Normalized DTW Cost")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


# =============================
# 4. DTW Alignment untuk file lama terbaik
# =============================
print("\nVisualisasi Warp Path (DTW Alignment) untuk file terbaik:")

best_match = df.sort_values('normalized_cost').iloc[0]
best_old_file = best_match['old_path']
best_old_label = best_match['old_label']

print(f"File lama terbaik: {best_old_file}")

m_old = old_mfcc[best_old_label][best_old_file]

# Karena TIDAK ada file baru, kita DTW-kan dirinya sendiri
D, wp = librosa.sequence.dtw(X=m_old, Y=m_old)

plt.figure(figsize=(6,6))
plt.imshow(D.T, origin='lower', aspect='auto', interpolation='nearest')
plt.plot([p[0] for p in wp], [p[1] for p in wp], color='yellow')
plt.title(f"DTW Cost Matrix & Warp Path\n{best_old_file} (Self Alignment)")
plt.xlabel("Time Index")
plt.ylabel("Time Index")
plt.show()



Menyiapkan visualisasi...


AttributeError: 'NoneType' object has no attribute 'groupby'